## Pipeline ENADE - Mineração de Dados Socioeconômicos
**TCC · Giovanna Naves Ribeiro · PUC Minas · 2025**
*Orientadora: Maria Inês Martins*

> Este notebook foi desenvolvido com auxílio do **Claude (Anthropic)**.

---

> **Download dos microdados:** https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enade
> Descompacte o ZIP e organize os arquivos em `data/{ano}/microdados{ano}_arq*.csv` (uma subpasta por ano), na MESMA pasta onde está este notebook.

### Sobre a unidade de análise deste pipeline
Este pipeline nunca cruza arquivos no nível do aluno: cada arquivo é primeiro resumido por curso (médias e proporções) e só depois essas agregações são unidas por `CO_CURSO`. Como resultado, cada linha do dado final representa **um curso em um ano**, não um aluno, o que ainda permite analisar perfil demográfico por curso, nota média por curso e a relação entre renda e nota comparando cursos entre si (inclusive por área).

### Etapa 1 - Instalação de dependências

In [1]:
!pip install pyarrow scikit-learn scipy seaborn tqdm --quiet

print("Dependências instaladas.")


Dependências instaladas.


### Etapa 2 - Diretórios do projeto
Os caminhos abaixo são sempre relativos à pasta onde este notebook está salvo, não dependem de usuário ou máquina. Basta que a pasta `data` esteja ao lado deste arquivo `.ipynb`.

In [ ]:
import os, json, logging, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("enade")

def _detectar_base_dir():
    candidatos = [Path.cwd()]
    val = globals().get("__vsc_ipynb_file__")  # VS Code define essa variável
    if val:
        candidatos.append(Path(val).resolve().parent)
    for cand in candidatos:
        if (cand / "data").exists():
            return cand.resolve()
    return candidatos[0].resolve()

BASE_DIR    = _detectar_base_dir()
DATA_DIR    = BASE_DIR / "data"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Pasta 'data' não encontrada em {BASE_DIR}.\n"
        "Coloque este notebook na mesma pasta que contém a pasta 'data' "
        "(com uma subpasta por ano dentro dela, ex.: data/2019, data/2021, ...)."
    )

print(f"Diretório base:   {BASE_DIR}")
print(f"Dados (entrada):  {DATA_DIR}")
print(f"Saídas (outputs): {OUTPUTS_DIR}")


📂 Diretório base:   C:\Users\GIOVANNA\TCC_ENADE
📂 Dados (entrada):  C:\Users\GIOVANNA\TCC_ENADE\data
📂 Saídas (outputs): C:\Users\GIOVANNA\TCC_ENADE\outputs


### Etapa 3 - Mapeamentos de variáveis, labels e parâmetros do questionário

In [ ]:

ARQUIVO_VARIAVEIS = {
    "arq1":  ["CO_IES","CO_CATEGAD","CO_ORGACAD","CO_GRUPO","CO_MODALIDADE",
              "CO_MUNIC_CURSO","CO_UF_CURSO","CO_REGIAO_CURSO"],
    "arq2":  ["ANO_IN_GRAD","CO_TURNO_GRADUACAO"],
    "arq3":  ["TP_PRES","NT_GER","NT_FG","NT_OBJ_FG","NT_DIS_FG",
              "NT_CE","NT_OBJ_CE","NT_DIS_CE"],
    "arq5":  ["TP_SEXO"],                 # sexo
    "arq6":  ["NU_IDADE"],                 # idade
    "arq8":  ["QE_I02"],                   # raça/cor
    "arq10": ["QE_I04"],                   # escolaridade do pai
    "arq11": ["QE_I05"],                   # escolaridade da mãe
    "arq14": ["QE_I08"],                   # renda familiar
    "arq16": ["QE_I10"],                   # situação de trabalho
    "arq17": ["QE_I11"],                   # financiamento do curso
    "arq18": ["QE_I12"],                   # tipo de auxílio/bolsa
    "arq21": ["QE_I15"],                   # cotas/reserva de vagas
    "arq23": ["QE_I17"],                   # tipo de escola no ensino médio
    "arq27": ["QE_I21"],                   # intercâmbio
    "arq28": ["QE_I22"],                   # livros lidos no ano
    "arq29": ["QE_I23"],                   # horas de estudo/semana
}

VARS_DESEMPENHO = ["NT_GER","NT_FG","NT_OBJ_FG","NT_DIS_FG","NT_CE","NT_OBJ_CE","NT_DIS_CE"]
# Para cada variável de desempenho, calculamos por curso tanto a MÉDIA quanto
# a MODA (nota mais frequente entre os alunos do curso) — por isso cada uma
# gera duas colunas no resultado final: NT_GER_MEDIA e NT_GER_MODA, etc.
VARS_DESEMPENHO_MEDIA = [f"{v}_MEDIA" for v in VARS_DESEMPENHO]
VARS_DESEMPENHO_MODA  = [f"{v}_MODA" for v in VARS_DESEMPENHO]

# Como cada variável deve ser resumida por curso na etapa de agregação:
VARS_NUMERICAS_MEDIA = [
    "NU_IDADE","ANO_IN_GRAD",
    "QE_I08_NUM","QE_I04_NUM","QE_I05_NUM","QE_I10_NUM","QE_I22_NUM","QE_I23_NUM",
]
# Variáveis categóricas: para cada uma, calculamos tanto a CONTAGEM ABSOLUTA
# de alunos por categoria (colunas QTD_...) quanto a proporção (usada
# internamente na clusterização e nos gráficos agregados).
VARS_CATEGORICAS_PROPORCAO = [
    "TP_SEXO", "QE_I02", "QE_I08", "QE_I04", "QE_I05", "QE_I10",
    "QE_I11", "QE_I12", "QE_I15", "QE_I17", "QE_I21",
]
VARS_IDENTIDADE_CURSO = [
    "CO_IES","CO_CATEGAD","CO_ORGACAD","CO_GRUPO","CO_MODALIDADE",
    "CO_MUNIC_CURSO","CO_UF_CURSO","CO_REGIAO_CURSO","CO_TURNO_GRADUACAO",
]

QE_AUSENTES     = ["8","9",8,9,""]
TP_PRES_REGULAR = 555

LABELS = {
    "TP_SEXO":    {"M":"Masculino","F":"Feminino"},
    "QE_I02":     {"A":"Branca","B":"Preta","C":"Amarela","D":"Parda","E":"Indígena","F":"Não declarado"},
    "QE_I08":     {"A":"Até 1,5 SM","B":"1,5–3 SM","C":"3–4,5 SM","D":"4,5–6 SM","E":"6–10 SM","F":"10–30 SM","G":">30 SM"},
    "QE_I08_ORD": {"A":1,"B":2,"C":3,"D":4,"E":5,"F":6,"G":7},
    "QE_I04":     {"A":"Nenhuma","B":"Fund.1–5","C":"Fund.6–9","D":"Médio","E":"Superior","F":"Pós-grad."},
    "QE_I04_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4,"F":5},
    "QE_I05":     {"A":"Nenhuma","B":"Fund.1–5","C":"Fund.6–9","D":"Médio","E":"Superior","F":"Pós-grad."},
    "QE_I05_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4,"F":5},
    "QE_I10":     {"A":"Não trabalha","B":"Eventualmente","C":"Até 20h","D":"21–39h","E":"40h+"},
    "QE_I10_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4},
    "QE_I17":     {"A":"Todo público","B":"Todo privado","C":"Exterior","D":"Maior p. público","E":"Maior p. privado","F":"BR+exterior"},
    "QE_I15":     {"A":"Não","B":"Étnico-racial","C":"Renda","D":"Escola pública","E":"Combinação","F":"Outro"},
    "QE_I11":     {"A":"Gratuito","B":"Nenhum (pago)","C":"ProUni integral","D":"ProUni parcial",
                   "E":"FIES","F":"ProUni+FIES","G":"Bolsa gov.","H":"Bolsa IES","I":"Bolsa outra",
                   "J":"Financ. IES","K":"Financ. bancário"},
    "QE_I12":     {"A":"Nenhum","B":"Moradia","C":"Alimentação","D":"Moradia+alim.","E":"Permanência","F":"Outro"},
    "QE_I21":     {"A":"Sim","B":"Não"},
    "QE_I22":     {"A":"Nenhum","B":"1–2","C":"3–5","D":"6–8","E":"Mais de 8"},
    "QE_I22_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4},
    "QE_I23":     {"A":"Nenhuma","B":"1–3h","C":"4–7h","D":"8–12h","E":"Mais de 12h"},
    "QE_I23_ORD": {"A":0,"B":1,"C":2,"D":3,"E":4},
    "CO_CATEGAD":  {1:"Pública Federal",2:"Pública Estadual",3:"Pública Municipal",
                    4:"Privada c/ fins luc.",5:"Privada s/ fins luc.",7:"Especial",8:"Comunitária"},
    "CO_MODALIDADE":{0:"EAD",1:"Presencial"},
}

CO_GRUPO_NOMES = {
    5:"Medicina Veterinária",6:"Odontologia",12:"Medicina",
    17:"Agronomia",19:"Farmácia",
    21:"Arquitetura e Urbanismo",23:"Enfermagem",27:"Fonoaudiologia",28:"Nutrição",36:"Fisioterapia",
    51:"Zootecnia",55:"Biomedicina",
    69:"Tecnologia em Radiologia",90:"Tecnologia em Agronegócios",91:"Tecnologia em Gestão Hospitalar",92:"Tecnologia em Gestão Ambiental",
    95:"Tecnologia em Estética e Cosmética",5710:"Engenharia Civil",5806:"Engenharia Elétrica",5814:"Engenharia de Controle e Automação",
    5902:"Engenharia Mecânica",6002:"Engenharia de Alimentos",6008:"Engenharia Química",
    6208:"Engenharia de Produção",6307:"Engenharia AMbiental",6405:"Engenharia Florestal",
    6410:"Tecnologia em Segurança do Trabalho",6411:"Engenharia de Computação I",
}


NOMES_QTD = {
    "TP_SEXO": {
        "M": "QTD_SEXO_MASCULINO", "F": "QTD_SEXO_FEMININO",
    },
    "QE_I02": {  # raça/cor
        "A": "QTD_RACA_BRANCA", "B": "QTD_RACA_PRETA", "C": "QTD_RACA_AMARELA",
        "D": "QTD_RACA_PARDA", "E": "QTD_RACA_INDIGENA", "F": "QTD_RACA_NAO_DECLARADO",
    },
    "QE_I08": {  # renda familiar
        "A": "QTD_RENDA_ATE_1_5_SM", "B": "QTD_RENDA_1_5_A_3_SM", "C": "QTD_RENDA_3_A_4_5_SM",
        "D": "QTD_RENDA_4_5_A_6_SM", "E": "QTD_RENDA_6_A_10_SM", "F": "QTD_RENDA_10_A_30_SM",
        "G": "QTD_RENDA_MAIS_DE_30_SM",
    },
    "QE_I04": {  # escolaridade do pai
        "A": "QTD_ESCOL_PAI_NENHUMA", "B": "QTD_ESCOL_PAI_FUND_1_5", "C": "QTD_ESCOL_PAI_FUND_6_9",
        "D": "QTD_ESCOL_PAI_MEDIO", "E": "QTD_ESCOL_PAI_SUPERIOR", "F": "QTD_ESCOL_PAI_POS_GRAD",
    },
    "QE_I05": {  # escolaridade da mãe
        "A": "QTD_ESCOL_MAE_NENHUMA", "B": "QTD_ESCOL_MAE_FUND_1_5", "C": "QTD_ESCOL_MAE_FUND_6_9",
        "D": "QTD_ESCOL_MAE_MEDIO", "E": "QTD_ESCOL_MAE_SUPERIOR", "F": "QTD_ESCOL_MAE_POS_GRAD",
    },
    "QE_I10": {  # situação de trabalho
        "A": "QTD_TRABALHO_NAO_TRABALHA", "B": "QTD_TRABALHO_EVENTUAL", "C": "QTD_TRABALHO_ATE_20H",
        "D": "QTD_TRABALHO_21_A_39H", "E": "QTD_TRABALHO_40H_OU_MAIS",
    },
    "QE_I11": {  # financiamento do curso
        "A": "QTD_FINANC_GRATUITO", "B": "QTD_FINANC_NENHUM_PAGO", "C": "QTD_FINANC_PROUNI_INTEGRAL",
        "D": "QTD_FINANC_PROUNI_PARCIAL", "E": "QTD_FINANC_FIES", "F": "QTD_FINANC_PROUNI_PARCIAL_FIES",
        "G": "QTD_FINANC_BOLSA_GOV", "H": "QTD_FINANC_BOLSA_IES", "I": "QTD_FINANC_BOLSA_OUTRA",
        "J": "QTD_FINANC_FINANCIAMENTO_IES", "K": "QTD_FINANC_FINANCIAMENTO_BANCARIO",
    },
    "QE_I12": {  # tipo de auxílio/bolsa
        "A": "QTD_AUXILIO_NENHUM", "B": "QTD_AUXILIO_MORADIA", "C": "QTD_AUXILIO_ALIMENTACAO",
        "D": "QTD_AUXILIO_MORADIA_E_ALIMENTACAO", "E": "QTD_AUXILIO_PERMANENCIA", "F": "QTD_AUXILIO_OUTRO",
    },
    "QE_I15": {  # cotas/reserva de vagas
        "A": "QTD_COTAS_NAO", "B": "QTD_COTAS_ETNICO_RACIAL", "C": "QTD_COTAS_RENDA",
        "D": "QTD_COTAS_ESCOLA_PUBLICA", "E": "QTD_COTAS_COMBINACAO_DE_CRITERIOS", "F": "QTD_COTAS_OUTRO_SISTEMA",
    },
    "QE_I17": {  # tipo de escola no ensino médio
        "A": "QTD_ESCOLA_EM_TODO_PUBLICO", "B": "QTD_ESCOLA_EM_TODO_PRIVADO", "C": "QTD_ESCOLA_EM_EXTERIOR",
        "D": "QTD_ESCOLA_EM_MAIOR_PARTE_PUBLICO", "E": "QTD_ESCOLA_EM_MAIOR_PARTE_PRIVADO",
        "F": "QTD_ESCOLA_EM_BRASIL_E_EXTERIOR",
    },
    "QE_I21": {  # intercâmbio
        "A": "QTD_INTERCAMBIO_SIM", "B": "QTD_INTERCAMBIO_NAO",
    },
}

print("Configuração carregada.")
print(f"   Arquivos de microdados usados: {list(ARQUIVO_VARIAVEIS.keys())}")


✅ Configuração carregada.
   Arquivos de microdados usados: ['arq1', 'arq2', 'arq3', 'arq5', 'arq6', 'arq8', 'arq10', 'arq11', 'arq14', 'arq16', 'arq17', 'arq18', 'arq21', 'arq23', 'arq27', 'arq28', 'arq29']


### Etapa 4 - Descoberta automática dos anos disponíveis
Todas as subpastas de `data/` nomeadas apenas com o ano (ex.: `data/2019`) são detectadas automaticamente. Não há mais um intervalo fixo `ANO_INICIO`/`ANO_FIM` para editar: para incluir um novo ano, basta adicionar a pasta correspondente.

In [ ]:

ANOS_DISPONIVEIS = sorted([
    int(p.name) for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
])

if not ANOS_DISPONIVEIS:
    raise FileNotFoundError(
        f"Nenhuma subpasta de ano encontrada em {DATA_DIR}.\n"
        "Crie uma subpasta por ano (ex.: data/2019) com os arquivos "
        "microdados{ano}_arq*.csv dentro."
    )

print(f"Anos encontrados ({len(ANOS_DISPONIVEIS)}): {ANOS_DISPONIVEIS}\n")

for ano in ANOS_DISPONIVEIS:
    pasta = DATA_DIR / str(ano)
    arquivos = [p for p in pasta.glob("*.*") if p.suffix.lower() in (".csv", ".txt")]
    print(f"   {ano}: {len(arquivos)} arquivos")


📅 Anos encontrados (13): [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023]

   2010: 27 arquivos
   2011: 27 arquivos
   2012: 27 arquivos
   2013: 31 arquivos
   2014: 42 arquivos
   2015: 32 arquivos
   2016: 32 arquivos
   2017: 42 arquivos
   2018: 32 arquivos
   2019: 32 arquivos
   2021: 43 arquivos
   2022: 33 arquivos
   2023: 32 arquivos


### Etapa 5 - Funções de ingestão e agregação por curso
Esta célula apenas **define** as funções; nenhum dado é processado ainda.

In [ ]:

NA_VALUES = ["", ".", "..", " ", "NA", "N/A"]


def _localizar_arquivo(sources, ano, nome_arq):
    candidatos = [
        f"microdados{ano}_{nome_arq}.csv", f"microdados{ano}_{nome_arq}.txt",
        f"microdados_{ano}_{nome_arq}.csv", f"microdados_{ano}_{nome_arq}.txt",
        f"{nome_arq}.csv", f"{nome_arq}.txt",
        f"MICRODADOS{ano}_{nome_arq.upper()}.CSV", f"MICRODADOS{ano}_{nome_arq.upper()}.TXT",
    ]
    return next((sources[c] for c in candidatos if c in sources), None)


def _ler_csv(csv_path, variaveis):
    """Lê um arquivo de microdados mantendo apenas NU_ANO, CO_CURSO e as variáveis pedidas."""
    cols_ok = ["NU_ANO", "CO_CURSO"] + variaveis
    for enc in ("latin-1", "utf-8", "utf-8-sig"):
        for quoting in (0, 3):  # 0=QUOTE_MINIMAL, 3=QUOTE_NONE
            try:
                df = pd.read_csv(
                    csv_path, sep=";", encoding=enc,
                    usecols=lambda c: c.strip() in cols_ok,
                    low_memory=False, quoting=quoting,
                    na_values=NA_VALUES, keep_default_na=False,
                )
                df.columns = df.columns.str.strip()
                for col in df.select_dtypes(include="object").columns:
                    df[col] = df[col].str.strip().str.strip('"')
                return df
            except Exception:
                continue
    return pd.DataFrame()


def _codificar_ordinais(df):
    """Converte códigos de letra (A,B,C...) em escala numérica ordinal, quando existir mapeamento."""
    df = df.copy()
    mapas = {"QE_I08": LABELS["QE_I08_ORD"], "QE_I04": LABELS["QE_I04_ORD"],
              "QE_I05": LABELS["QE_I05_ORD"], "QE_I10": LABELS["QE_I10_ORD"],
              "QE_I22": LABELS["QE_I22_ORD"], "QE_I23": LABELS["QE_I23_ORD"]}
    for col, mapa in mapas.items():
        if col in df.columns:
            df[f"{col}_NUM"] = df[col].astype(str).str.strip().str.upper().map(mapa)
    return df


def _tratar_ausentes_qe(df):
    df = df.copy()
    qe_cols = [c for c in df.columns if c.startswith("QE_I")]
    ausentes_str = [str(v) for v in QE_AUSENTES]
    for col in qe_cols:
        mask = df[col].astype(str).str.strip().isin(ausentes_str)
        df.loc[mask, col] = np.nan
    return df


def _moda(serie):
    """Valor mais frequente de uma série (ignora NaN). None se a série for vazia."""
    m = serie.mode()
    return m.iloc[0] if not m.empty else np.nan


def agregar_arquivo_por_curso(df_arq, nome_arq):
    """
    Resume um arquivo de microdados (1 linha = 1 respondente) em
    1 linha por (NU_ANO, CO_CURSO):
      - MÉDIA e MODA para as notas (NT_GER_MEDIA / NT_GER_MODA, etc.)
      - MÉDIA para variáveis ordinais convertidas em escala numérica
        (QE_I08_NUM, QE_I04_NUM, ...) — usadas em correlação/clusterização
      - CONTAGEM ABSOLUTA (QTD_...) e PROPORÇÃO (..._PCT_...) para
        variáveis categóricas (sexo, raça, renda, financiamento, etc.)
      - MODA para atributos de identidade do curso (praticamente
        constantes dentro do mesmo CO_CURSO, ex.: CO_GRUPO, CO_UF_CURSO)
    """
    if df_arq.empty or "CO_CURSO" not in df_arq.columns:
        return pd.DataFrame()

    df = df_arq.copy()

    if nome_arq == "arq3":
        if "TP_PRES" in df.columns:
            df = df[df["TP_PRES"] == TP_PRES_REGULAR]
        for col in VARS_DESEMPENHO:
            if col in df.columns:
                df[col] = (df[col].astype(str).str.replace(",", ".", regex=False)
                           .pipe(pd.to_numeric, errors="coerce"))

    df = _tratar_ausentes_qe(df)
    df = _codificar_ordinais(df)

    for col in VARS_CATEGORICAS_PROPORCAO:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()

    grp = df.groupby(["NU_ANO", "CO_CURSO"])
    partes = [grp.size().rename(f"N_{nome_arq}")]

    for col in df.columns:
        if col in ("NU_ANO", "CO_CURSO"):
            continue

        if col in VARS_DESEMPENHO:
            # Notas: guardamos média E moda separadamente.
            partes.append(grp[col].mean().rename(f"{col}_MEDIA"))
            partes.append(grp[col].agg(_moda).rename(f"{col}_MODA"))

        elif col in VARS_NUMERICAS_MEDIA:
            partes.append(grp[col].mean().rename(col))

        elif col in VARS_CATEGORICAS_PROPORCAO:
            contagem = (df.groupby(["NU_ANO", "CO_CURSO"])[col]
                          .value_counts().unstack(fill_value=0))
            total = contagem.sum(axis=1)
            proporcao = contagem.div(total.replace(0, np.nan), axis=0)
            proporcao.columns = [f"{col}_PCT_{c}" for c in proporcao.columns]
            nomes_amigaveis = NOMES_QTD.get(col, {})
            contagem.columns = [nomes_amigaveis.get(c, f"QTD_{col}_{c}") for c in contagem.columns]
            partes.append(contagem)
            partes.append(proporcao)

        elif col in VARS_IDENTIDADE_CURSO:
            partes.append(grp[col].agg(_moda).rename(col))
        # demais colunas (ex.: a versão bruta em letra de QE_I08, já convertida
        # para QE_I08_NUM acima) não são agregadas individualmente.

    return pd.concat(partes, axis=1).reset_index()


def ingerir_e_agregar(ano):
    """
    Para um ano: lê cada arquivo de microdados disponível, agrega por curso
    e junta as agregações por (NU_ANO, CO_CURSO).
    Retorna um DataFrame em que CADA LINHA É UM CURSO (não um aluno).
    """
    pasta = DATA_DIR / str(ano)
    sources = {p.name: p for p in pasta.glob("*.*") if p.suffix.lower() in (".csv", ".txt")}
    if not sources:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {pasta}")

    log.info(f"[Ingestor] ENADE {ano} — {len(sources)} arquivos disponíveis")

    df_curso, lidos, ausentes = None, 0, []
    for nome_arq, variaveis in ARQUIVO_VARIAVEIS.items():
        caminho = _localizar_arquivo(sources, ano, nome_arq)
        if caminho is None:
            ausentes.append(nome_arq)
            continue
        df_arq = _ler_csv(caminho, variaveis)
        if df_arq.empty:
            continue
        df_arq["NU_ANO"] = ano

        df_agg = agregar_arquivo_por_curso(df_arq, nome_arq)
        if df_agg.empty:
            continue
        lidos += 1
        df_curso = df_agg if df_curso is None else df_curso.merge(
            df_agg, on=["NU_ANO", "CO_CURSO"], how="outer")

    if df_curso is None or df_curso.empty:
        raise ValueError(f"Nenhum arquivo agregado para {ano}.")
    if ausentes:
        log.warning(f"[Ingestor] Arquivos não encontrados (ignorados): {', '.join(ausentes)}")

    if "N_arq3" in df_curso.columns:
        df_curso = df_curso.rename(columns={"N_arq3": "N_ALUNOS"})

    # Colunas de contagem absoluta (QTD_...) nunca devem ficar em branco:
    # se um curso não tiver dados naquele arquivo específico, a contagem é 0.
    cols_qtd = [c for c in df_curso.columns if c.startswith("QTD_")]
    if cols_qtd:
        df_curso[cols_qtd] = df_curso[cols_qtd].fillna(0).astype(int)

    log.info(f"[Ingestor] {lidos} arquivos agregados → {len(df_curso):,} cursos × {df_curso.shape[1]} colunas")
    return df_curso


print("Funções de ingestão/agregação por curso definidas.")


✅ Funções de ingestão/agregação por curso definidas.


### Etapa 6 - Teste de ingestão para um ano (verificação do `df`)

In [ ]:
ANO_TESTE = ANOS_DISPONIVEIS[-1]

print(f"Testando ingestão/agregação para {ANO_TESTE}...")
df_curso_bruto = ingerir_e_agregar(ANO_TESTE)

print(f"\nShape (cursos × colunas): {df_curso_bruto.shape}")
print(f"Colunas (amostra): {list(df_curso_bruto.columns[:15])}")
df_curso_bruto.head(5)


[Ingestor] ENADE 2023 — 32 arquivos disponíveis


🔍 Testando ingestão/agregação para 2023...


[Ingestor] 17 arquivos agregados → 9,812 cursos × 198 colunas



Shape (cursos × colunas): (9812, 198)
Colunas (amostra): ['NU_ANO', 'CO_CURSO', 'N_arq1', 'CO_IES', 'CO_CATEGAD', 'CO_ORGACAD', 'CO_GRUPO', 'CO_MODALIDADE', 'CO_MUNIC_CURSO', 'CO_UF_CURSO', 'CO_REGIAO_CURSO', 'N_arq2', 'ANO_IN_GRAD', 'CO_TURNO_GRADUACAO', 'N_ALUNOS']


,NU_ANO,CO_CURSO,N_arq1,CO_IES,CO_CATEGAD,CO_ORGACAD,CO_GRUPO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO,...,QTD_INTERCAMBIO_SIM,QTD_INTERCAMBIO_NAO,QTD_QE_I21_NAN,QE_I21_PCT_A,QE_I21_PCT_B,QE_I21_PCT_NAN,N_arq28,QE_I22_NUM,N_arq29,QE_I23_NUM
0,2023,3,33,1,1,10028,5710,1,5103403,51,...,27,4,2,0.818182,0.121212,0.060606,33,1.451613,33,2.516129
1,2023,9,42,1,1,10028,17,1,5103403,51,...,36,6,0,0.857143,0.142857,0.000000,42,1.166667,42,2.095238
2,2023,10,13,1,1,10028,6405,1,5103403,51,...,9,3,1,0.692308,0.230769,0.076923,13,1.416667,13,1.666667
3,2023,12,78,1,1,10028,12,1,5103403,51,...,73,5,0,0.935897,0.064103,0.000000,78,1.358974,78,2.910256
4,2023,16,28,1,1,10028,5806,1,5103403,51,...,23,4,1,0.821429,0.142857,0.035714,28,1.629630,28,2.481481


### Etapa 7 - Engenharia de atributos por curso (verificação do `df`)

In [ ]:
def adicionar_derivadas_curso(df):
    """Cria colunas derivadas a partir das médias/proporções já calculadas por curso."""
    df = df.copy()

    if "CO_GRUPO" in df.columns:
        df["NOME_CURSO"] = df["CO_GRUPO"].map(CO_GRUPO_NOMES).fillna(
            df["CO_GRUPO"].astype(str).apply(lambda x: f"Grupo {x}"))

    if "QE_I08_NUM" in df.columns:
        conds = [df["QE_I08_NUM"] <= 2, df["QE_I08_NUM"].between(3, 4), df["QE_I08_NUM"] >= 5]
        df["FAIXA_RENDA_SIMPL"] = np.select(
            conds, ["Baixa (≤3 SM)", "Média (3–6 SM)", "Alta (>6 SM)"], default="Não informado")

    comps = []
    for col, div in [("QE_I04_NUM", 5), ("QE_I05_NUM", 5), ("QE_I23_NUM", 4), ("QE_I22_NUM", 4)]:
        if col in df.columns:
            comps.append(df[col] / div)
    if comps:
        df["IDX_CAPITAL_CULT"] = pd.concat(comps, axis=1).mean(axis=1)

    if "CO_CATEGAD" in df.columns:
        df["CATEGORIA_IES"] = df["CO_CATEGAD"].map(LABELS["CO_CATEGAD"])
    if "CO_MODALIDADE" in df.columns:
        df["MODALIDADE"] = df["CO_MODALIDADE"].map(LABELS["CO_MODALIDADE"])

    if "TP_SEXO_PCT_F" in df.columns:
        df["PCT_FEMININO"] = (df["TP_SEXO_PCT_F"] * 100).round(1)

    return df


df_curso_eng = adicionar_derivadas_curso(df_curso_bruto)

derivadas = ["NOME_CURSO", "FAIXA_RENDA_SIMPL", "IDX_CAPITAL_CULT",
             "CATEGORIA_IES", "MODALIDADE", "PCT_FEMININO"]
print("Variáveis derivadas criadas:", [c for c in derivadas if c in df_curso_eng.columns])
print(f"Shape final: {df_curso_eng.shape}  (cada linha = 1 curso em {ANO_TESTE})")
df_curso_eng[[c for c in derivadas if c in df_curso_eng.columns] + ["NT_GER_MEDIA", "NT_GER_MODA"]].dropna(subset=["NT_GER_MEDIA"]).head(5)


Variáveis derivadas criadas: ['NOME_CURSO', 'FAIXA_RENDA_SIMPL', 'IDX_CAPITAL_CULT', 'CATEGORIA_IES', 'MODALIDADE', 'PCT_FEMININO']
Shape final: (9812, 204)  (cada linha = 1 curso em 2023)


,NOME_CURSO,FAIXA_RENDA_SIMPL,IDX_CAPITAL_CULT,CATEGORIA_IES,MODALIDADE,PCT_FEMININO,NT_GER_MEDIA,NT_GER_MODA
0,Engenharia Civil,Média (3–6 SM),0.606048,Pública Federal,Presencial,21.2,59.612903,31.0
1,Agronomia,Média (3–6 SM),0.512202,Pública Federal,Presencial,31.0,59.377778,60.6
2,Engenharia Florestal,Não informado,0.509375,Pública Federal,Presencial,69.2,45.709091,43.6
3,Medicina,Não informado,0.632853,Pública Federal,Presencial,52.6,69.443590,61.5
4,Engenharia Elétrica,Média (3–6 SM),0.579167,Pública Federal,Presencial,21.4,51.273913,45.3


### Etapa 8 - Análise exploratória (nível curso)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ANÁLISE EXPLORATÓRIA — nível curso
# ════════════════════════════════════════════════════════════════════════════
# Todos os gráficos abaixo descrevem o conjunto de CURSOS (cada ponto/barra
# resume um curso), não mais alunos individuais.

def plot_distribuicao_notas(df):
    vars_nota = [v for v in ["NT_GER_MEDIA", "NT_FG_MEDIA", "NT_CE_MEDIA"] if v in df.columns]
    if not vars_nota:
        return
    fig, axes = plt.subplots(1, len(vars_nota), figsize=(5 * len(vars_nota), 4))
    if len(vars_nota) == 1:
        axes = [axes]
    cores = ["#0F7B6C", "#0D2137", "#F59E0B"]
    for ax, col, cor in zip(axes, vars_nota, cores):
        dados = df[col].dropna()
        titulo = col.replace("_MEDIA", "")
        ax.hist(dados, bins=40, color=cor, alpha=.8, edgecolor="white")
        ax.axvline(dados.mean(), color="red", ls="--", lw=1.5, label=f"Média: {dados.mean():.1f}")
        ax.set(title=f"{titulo} (média por curso)", xlabel="Nota média do curso", ylabel="Nº de cursos")
        ax.legend(fontsize=9)
    plt.suptitle("Distribuição das Notas Médias por Curso", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "dist_notas_por_curso.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def plot_nota_por_renda(df):
    if "FAIXA_RENDA_SIMPL" not in df.columns or "NT_GER_MEDIA" not in df.columns:
        return
    agg = (df.dropna(subset=["NT_GER_MEDIA"])
             .groupby("FAIXA_RENDA_SIMPL")["NT_GER_MEDIA"]
             .agg(["mean", "std", "count"]).reset_index())
    ordem = ["Baixa (≤3 SM)", "Média (3–6 SM)", "Alta (>6 SM)", "Não informado"]
    agg["ordem"] = agg["FAIXA_RENDA_SIMPL"].apply(lambda x: ordem.index(x) if x in ordem else 99)
    agg = agg.sort_values("ordem")
    fig, ax = plt.subplots(figsize=(9, 5))
    cores = sns.color_palette("YlOrRd", n_colors=len(agg))
    bars = ax.bar(agg["FAIXA_RENDA_SIMPL"], agg["mean"],
                   yerr=agg["std"] / np.sqrt(agg["count"]), color=cores, capsize=4)
    ax.set(ylabel="NT_GER média (entre cursos)",
           title="Nota Geral Média por Faixa de Renda Predominante do Curso")
    ax.grid(axis="y", alpha=.4)
    for bar, media, n in zip(bars, agg["mean"], agg["count"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + .3,
                 f"{media:.1f}\n(n={n} cursos)", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "nota_por_renda_curso.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()
    if "QE_I08_NUM" in df.columns:
        rho, pv = spearmanr(df["QE_I08_NUM"], df["NT_GER_MEDIA"], nan_policy="omit")
        print(f"\nSpearman (entre cursos) Renda × NT_GER: ρ={rho:.4f}, p={pv:.2e}")


def plot_top_cursos(df, top_n=15, min_cursos=5):
    if "NOME_CURSO" not in df.columns or "NT_GER_MEDIA" not in df.columns:
        return
    notas = (df.groupby("NOME_CURSO")["NT_GER_MEDIA"].agg(["mean", "count"])
                .query(f"count >= {min_cursos}").sort_values("mean", ascending=False)
                .head(top_n).reset_index())
    if notas.empty:
        print(f"Nenhuma área com pelo menos {min_cursos} cursos para o ranking.")
        return
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(notas["NOME_CURSO"][::-1], notas["mean"][::-1],
            color=sns.color_palette("Blues_r", n_colors=len(notas))[::-1])
    ax.set(xlabel="NT_GER média (entre cursos/instituições)",
           title=f"Top {top_n} Áreas por Nota Geral Média (mín. {min_cursos} cursos)")
    for i, (med, n) in enumerate(zip(notas["mean"][::-1], notas["count"][::-1])):
        ax.text(med + .2, i, f"{med:.1f} (n={n} cursos)", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "top_cursos_nota.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def plot_perfil_demografico(df):
    """Perfil demográfico agregado — sexo ponderado pelo nº de alunos de cada
    curso; raça/etnia como média simples das proporções por curso."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    if "TP_SEXO_PCT_F" in df.columns:
        sub = df[["TP_SEXO_PCT_F"]].copy()
        peso = df["N_ALUNOS"] if "N_ALUNOS" in df.columns else pd.Series(1, index=df.index)
        sub = sub.join(peso.rename("PESO")).dropna()
        if not sub.empty and sub["PESO"].sum() > 0:
            pct_f = np.average(sub["TP_SEXO_PCT_F"], weights=sub["PESO"])
            axes[0].pie([pct_f, 1 - pct_f], labels=["Feminino", "Masculino"], autopct="%1.1f%%",
                        colors=["#F59E0B", "#0F7B6C"], startangle=90)
            axes[0].set_title("Distribuição por Sexo\n(ponderado por nº de alunos)", fontweight="bold")

    cols_raca = [c for c in df.columns if c.startswith("QE_I02_PCT_")]
    if cols_raca:
        medias = df[cols_raca].mean().sort_values(ascending=True)
        labels_raca = {f"QE_I02_PCT_{k}": v for k, v in LABELS["QE_I02"].items()}
        axes[1].barh([labels_raca.get(i, i) for i in medias.index], medias.values * 100,
                     color=sns.color_palette("Blues_r", n_colors=len(medias)))
        axes[1].set(title="Raça/Etnia — % médio por curso", xlabel="% médio dos cursos")

    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / "perfil_demografico_curso.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


print("Funções de análise exploratória (nível curso) definidas.")


✅ Funções de análise exploratória (nível curso) definidas.


In [9]:
print("=== Distribuição das Notas Médias por Curso ===")
plot_distribuicao_notas(df_curso_eng)

print("\n=== Nota Média por Faixa de Renda do Curso ===")
plot_nota_por_renda(df_curso_eng)

print("\n=== Top Áreas por Nota Média ===")
plot_top_cursos(df_curso_eng, top_n=15)

print("\n=== Perfil Demográfico (agregado) ===")
plot_perfil_demografico(df_curso_eng)


=== Distribuição das Notas Médias por Curso ===

=== Nota Média por Faixa de Renda do Curso ===

Spearman (entre cursos) Renda × NT_GER: ρ=0.0895, p=3.92e-18

=== Top Áreas por Nota Média ===

=== Perfil Demográfico (agregado) ===


### Etapa 9 - Correlação socioeconômica × desempenho, entre cursos

In [ ]:
def correlacao_geral_curso(df):
    vars_soc = {"QE_I08_NUM": "Renda", "QE_I04_NUM": "Escol.Pai", "QE_I05_NUM": "Escol.Mãe",
                "QE_I10_NUM": "Trabalho", "QE_I22_NUM": "Leitura", "QE_I23_NUM": "H.Estudo",
                "IDX_CAPITAL_CULT": "Cap.Cultural"}
    vars_nota = [v for v in VARS_DESEMPENHO_MEDIA if v in df.columns]
    rows = []
    for col_s, label in vars_soc.items():
        if col_s not in df.columns:
            continue
        row = {"Variável": label}
        for vn in vars_nota:
            par = df[[col_s, vn]].dropna()
            if len(par) < 10:
                row[vn] = np.nan
                continue
            rho, pv = spearmanr(par[col_s], par[vn])
            row[vn] = round(rho, 4)
            row[f"{vn}_p"] = round(pv, 6)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Variável")


def correlacao_por_area(df, var_soc="QE_I08_NUM", var_nota="NT_GER_MEDIA", min_cursos=10):
    """Correlação Renda × Nota calculada ENTRE CURSOS, separadamente para cada área (CO_GRUPO)."""
    if var_soc not in df.columns or var_nota not in df.columns or "CO_GRUPO" not in df.columns:
        return pd.DataFrame()
    rows = []
    for grupo, sub in df.groupby("CO_GRUPO"):
        sub = sub[[var_soc, var_nota]].dropna()
        if len(sub) < min_cursos:
            continue
        rho, pv = spearmanr(sub[var_soc], sub[var_nota])
        rows.append({"CO_GRUPO": grupo, "NOME_CURSO": CO_GRUPO_NOMES.get(grupo, f"Grupo {grupo}"),
                     "N_CURSOS": len(sub), "SPEARMAN_RHO": round(rho, 4), "P_VALOR": round(pv, 6),
                     "SIGNIFICATIVO": "Sim" if pv < 0.05 else "Não"})
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values("SPEARMAN_RHO", ascending=False)


def heatmap_correlacoes_curso(df, salvar=True):
    interesse = ["QE_I08_NUM", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM",
                 "QE_I22_NUM", "QE_I23_NUM", "IDX_CAPITAL_CULT",
                 "NT_GER_MEDIA", "NT_FG_MEDIA", "NT_CE_MEDIA"]
    labs = {"QE_I08_NUM": "Renda", "QE_I04_NUM": "Escol.Pai", "QE_I05_NUM": "Escol.Mãe",
            "QE_I10_NUM": "Trabalho", "QE_I22_NUM": "Leitura", "QE_I23_NUM": "H.Estudo",
            "IDX_CAPITAL_CULT": "Cap.Cultural", "NT_GER_MEDIA": "Nota Geral",
            "NT_FG_MEDIA": "Form.Geral", "NT_CE_MEDIA": "Comp.Espec."}
    cols = [v for v in interesse if v in df.columns]
    if len(cols) < 2:
        return
    sub = df[cols].dropna(thresh=max(1, len(cols) // 2))
    corr = sub.corr(method="spearman").rename(index=labs, columns=labs)
    fig, ax = plt.subplots(figsize=(10, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
                center=0, vmin=-1, vmax=1, linewidths=.5, ax=ax, cbar_kws={"label": "Spearman ρ"})
    ax.set_title("Correlação Spearman entre Cursos\n(perfil socioeconômico médio × desempenho médio)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    if salvar:
        plt.savefig(OUTPUTS_DIR / "heatmap_correlacoes_curso.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def barplot_correlacao_areas(df_corr, salvar=True):
    if df_corr.empty:
        return
    dp = df_corr.sort_values("SPEARMAN_RHO")
    cores = ["#DC2626" if r < 0 else "#0F7B6C" for r in dp["SPEARMAN_RHO"]]
    fig, ax = plt.subplots(figsize=(10, max(6, len(dp) * .4)))
    bars = ax.barh(dp["NOME_CURSO"], dp["SPEARMAN_RHO"], color=cores)
    ax.axvline(0, color="black", lw=.8)
    ax.set(xlabel="Spearman ρ (Renda × NT_GER, entre cursos da área)",
           title="Correlação Renda × Nota por Área (entre cursos)")
    ax.grid(axis="x", alpha=.3)
    for bar, rho in zip(bars, dp["SPEARMAN_RHO"]):
        ax.text(bar.get_width() + .01 * np.sign(rho + 1e-9), bar.get_y() + bar.get_height() / 2,
                 f"{rho:.3f}", va="center", fontsize=9)
    plt.tight_layout()
    if salvar:
        plt.savefig(OUTPUTS_DIR / "barplot_correlacao_areas.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def analisar_correlacoes(df):
    log.info("[Correlação] Calculando (nível curso)...")
    geral = correlacao_geral_curso(df)
    geral.to_csv(OUTPUTS_DIR / "correlacao_geral.csv", sep=";", decimal=",")
    por_area = correlacao_por_area(df)
    if not por_area.empty:
        por_area.to_csv(OUTPUTS_DIR / "correlacao_por_area.csv", index=False, sep=";", decimal=",")
        barplot_correlacao_areas(por_area)
    heatmap_correlacoes_curso(df)
    return geral, por_area


print("Funções de correlação (nível curso) definidas.")


✅ Funções de correlação (nível curso) definidas.


In [11]:
df_corr_geral, df_corr_area = analisar_correlacoes(df_curso_eng)

print("\n=== Correlação Spearman (geral, entre cursos) ===")
cols_nota = [v for v in ["NT_GER_MEDIA", "NT_FG_MEDIA", "NT_CE_MEDIA"] if v in df_corr_geral.columns]
print(df_corr_geral[cols_nota].to_string())

print("\n=== Correlação por Área (Renda × NT_GER, entre cursos da área) ===")
if not df_corr_area.empty:
    print(df_corr_area[["NOME_CURSO", "N_CURSOS", "SPEARMAN_RHO", "P_VALOR", "SIGNIFICATIVO"]].to_string(index=False))
else:
    print("Poucos cursos por área neste ano para calcular correlação separada.")


[Correlação] Calculando (nível curso)...



=== Correlação Spearman (geral, entre cursos) ===
              NT_GER_MEDIA  NT_FG_MEDIA  NT_CE_MEDIA
Variável                                            
Renda               0.0895       0.3496       0.0334
Escol.Pai           0.2725       0.4617       0.2124
Escol.Mãe           0.3130       0.4676       0.2552
Trabalho           -0.5749      -0.3006      -0.5790
Leitura             0.1885       0.1525       0.1799
H.Estudo            0.4239       0.4656       0.3810
Cap.Cultural        0.4200       0.5473       0.3583

=== Correlação por Área (Renda × NT_GER, entre cursos da área) ===
                         NOME_CURSO  N_CURSOS  SPEARMAN_RHO  P_VALOR SIGNIFICATIVO
               Engenharia Florestal        59        0.4718 0.000162           Sim
                       Fisioterapia       699        0.4395 0.000000           Sim
                           Nutrição       601        0.4332 0.000000           Sim
 Engenharia de Controle e Automação       151        0.4264 0.000000    

### Etapa 10 - Clusterização de perfis demográficos por curso (bloco independente)

Este bloco funciona de forma independente do restante do notebook — basta um DataFrame em que cada linha seja um curso (como `df_curso_eng`), já com renda média, % por sexo/raça e nota média.

O pipeline:
1. Constrói a matriz de features socioeconômicas/demográficas por curso, normalizada
2. Determina o k ótimo pelo **Método do Cotovelo** + Silhouette
3. Executa **K-Means** e gera visualização **PCA**
4. Caracteriza cada cluster pelo perfil médio dos cursos que o compõem
5. Identifica o cluster dominante por área (`CO_GRUPO`)

> Altere `k=None` para automático, ou defina `k=4` para fixar o número de clusters.

In [ ]:

# Este bloco é independente do resto do notebook: para rodar, basta ter um
# DataFrame em que cada linha é um curso (como df_curso_eng), com as colunas
# de renda média, % por sexo/raça e nota média já calculadas pelas etapas
# anteriores.
#
# Os clusters agrupam CURSOS que têm perfil socioeconômico/demográfico
# parecido (renda média, % feminino, % por raça, nota média etc.)

def construir_matriz_cluster(df):
    candidatas = ["QE_I08_NUM", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM", "IDX_CAPITAL_CULT", "NT_GER_MEDIA"]
    candidatas += [c for c in df.columns if c.startswith("TP_SEXO_PCT_")]
    candidatas += [c for c in df.columns if c.startswith("QE_I02_PCT_")]
    cols = [c for c in candidatas if c in df.columns]
    if not cols:
        raise ValueError("Nenhuma feature disponível para clusterização.")
    X = df[cols].copy()
    X = X.dropna(thresh=max(1, int(.6 * X.shape[1])))
    X = X.fillna(X.median())
    scaler = MinMaxScaler()
    X_sc = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    log.info(f"[Cluster] Matriz: {X_sc.shape[0]:,} cursos × {X_sc.shape[1]} features")
    return X_sc, cols


def elbow_method(X, k_min=2, k_max=8, random_state=42, salvar=True):
    k_max_possivel = min(k_max, len(X) - 1)
    ks = list(range(k_min, k_max_possivel + 1))
    if len(ks) < 2:
        log.warning("[Cluster] Poucos cursos para Elbow Method — usando k=2.")
        return max(2, len(ks))

    inercias, sils = [], []
    for k in ks:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        lb = km.fit_predict(X)
        inercias.append(km.inertia_)
        sils.append(silhouette_score(X, lb) if k > 1 else 0)
    k_otimo = ks[int(np.argmax(sils))]
    log.info(f"[Cluster] k ótimo: {k_otimo}  Silhouettes: {dict(zip(ks, np.round(sils, 3)))}")

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(ks, inercias, "o-", color="#0F7B6C", lw=2)
    a1.axvline(k_otimo, color="#F59E0B", ls="--", label=f"k={k_otimo}")
    a1.set(xlabel="k", ylabel="Inércia (WCSS)", title="Método do Cotovelo"); a1.legend(); a1.grid(alpha=.3)
    a2.plot(ks, sils, "s-", color="#0D2137", lw=2)
    a2.axvline(k_otimo, color="#F59E0B", ls="--")
    a2.set(xlabel="k", ylabel="Silhouette Score", title="Silhouette por k"); a2.grid(alpha=.3)
    plt.tight_layout()
    if salvar:
        plt.savefig(OUTPUTS_DIR / "elbow.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()
    return k_otimo


def kmeans_cluster(X, k, random_state=42):
    log.info(f"[Cluster] K-Means k={k}...")
    km = KMeans(n_clusters=k, random_state=random_state, n_init=20, max_iter=500)
    labels = km.fit_predict(X)
    dist = {int(u): int(c) for u, c in zip(*np.unique(labels, return_counts=True))}
    log.info(f"[Cluster] Distribuição (cursos por cluster): {dist}")
    return labels


def dbscan_cluster(X, eps=0.2, min_samples=10):
    log.info(f"[Cluster] DBSCAN eps={eps}, min_samples={min_samples}...")
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
    n_cl = len(set(labels)) - (1 if -1 in labels else 0)
    log.info(f"[Cluster] {n_cl} clusters, {(labels == -1).sum():,} cursos classificados como ruído")
    return labels


def pca_visualizacao(X, labels, salvar=True):
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)
    ve = pca.explained_variance_ratio_
    log.info(f"[Cluster] PCA: PC1={ve[0]:.1%}, PC2={ve[1]:.1%}")
    dp = pd.DataFrame(coords, columns=["PC1", "PC2"], index=X.index)
    dp["CLUSTER"] = labels
    pal = sns.color_palette("tab10", n_colors=dp["CLUSTER"].nunique())
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, cl in enumerate(sorted(dp["CLUSTER"].unique())):
        m = dp["CLUSTER"] == cl
        ax.scatter(dp.loc[m, "PC1"], dp.loc[m, "PC2"], c=[pal[i % len(pal)]],
                   label=f"Cluster {cl}" if cl >= 0 else "Ruído", alpha=.6, s=18)
    ax.set(xlabel=f"PC1 ({ve[0]:.1%})", ylabel=f"PC2 ({ve[1]:.1%})",
           title="Clusters de Cursos por Perfil Demográfico (PCA)")
    ax.legend(bbox_to_anchor=(1.05, 1)); ax.grid(alpha=.3)
    plt.tight_layout()
    if salvar:
        plt.savefig(OUTPUTS_DIR / "pca_clusters.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()


def caracterizar_clusters(df, features):
    vars_num = list(dict.fromkeys(
        [c for c in features if c in df.columns] + [v for v in VARS_DESEMPENHO_MEDIA if v in df.columns]
    ))
    perfil = df.groupby("CLUSTER_KMEANS")[vars_num].mean().round(3)
    perfil.insert(0, "N_CURSOS", df.groupby("CLUSTER_KMEANS").size())
    perfil.insert(1, "PCT", (perfil["N_CURSOS"] / len(df) * 100).round(1))
    return perfil


def perfil_por_area(df, top_n=20):
    if "CO_GRUPO" not in df.columns:
        return pd.DataFrame()
    rows = []
    for grupo, sub in df.groupby("CO_GRUPO"):
        vc = sub["CLUSTER_KMEANS"].value_counts()
        if vc.empty:
            continue
        rows.append({
            "CO_GRUPO": grupo, "NOME_CURSO": CO_GRUPO_NOMES.get(grupo, f"Grupo {grupo}"),
            "N_CURSOS": len(sub), "CLUSTER_DOMINANTE": int(vc.index[0]),
            "PCT_DOMINANTE": round(vc.iloc[0] / len(sub) * 100, 1),
            "NT_GER_MEDIA": round(sub["NT_GER_MEDIA"].mean(), 2) if "NT_GER_MEDIA" in sub.columns else np.nan,
        })
    if not rows:
        return pd.DataFrame()
    return (pd.DataFrame(rows).sort_values("N_CURSOS", ascending=False)
              .reset_index(drop=True).head(top_n))


def clusterizar(df, k=None, metodo="kmeans", random_state=42):
    """
    Pipeline completo de clusterização por curso.
    Parâmetros: k=None (automático via Elbow), metodo='kmeans'|'dbscan'|'ambos'
    Retorna (df_com_cluster, perfil_clusters, perfil_por_area)
    """
    X, features = construir_matriz_cluster(df)
    df_al = df.loc[X.index].copy()
    if k is None:
        k = elbow_method(X)
    labels_km = kmeans_cluster(X, k=k, random_state=random_state)
    df_al["CLUSTER_KMEANS"] = labels_km
    if metodo in ("dbscan", "ambos"):
        df_al["CLUSTER_DBSCAN"] = dbscan_cluster(X)
    pca_visualizacao(X, labels_km)
    perfil = caracterizar_clusters(df_al, features)
    perfil.to_csv(OUTPUTS_DIR / "perfil_clusters.csv", sep=";", decimal=",")
    pa = perfil_por_area(df_al)
    if not pa.empty:
        pa.to_csv(OUTPUTS_DIR / "perfil_por_area.csv", index=False, sep=";", decimal=",")
    return df_al, perfil, pa


print("Funções de clusterização (perfil demográfico por curso) definidas.")


✅ Funções de clusterização (perfil demográfico por curso) definidas.


In [13]:
df_cl, perfil_clusters, perfil_areas = clusterizar(
    df_curso_eng,
    k=None,           # None = automático via Elbow Method
    metodo="kmeans",  # "kmeans" | "dbscan" | "ambos"
)

print("\n=== Perfil dos Clusters (médias por curso dentro de cada cluster) ===")
print(perfil_clusters.to_string())

if not perfil_areas.empty:
    print("\n=== Cluster Dominante por Área ===")
    print(perfil_areas.to_string(index=False))

df_cl.head(5)


[Cluster] Matriz: 9,812 cursos × 16 features
[Cluster] k ótimo: 2  Silhouettes: {2: np.float64(0.324), 3: np.float64(0.238), 4: np.float64(0.256), 5: np.float64(0.215), 6: np.float64(0.201), 7: np.float64(0.219), 8: np.float64(0.21)}
[Cluster] K-Means k=2...
[Cluster] Distribuição (cursos por cluster): {0: 6242, 1: 3570}
[Cluster] PCA: PC1=42.7%, PC2=21.1%



=== Perfil dos Clusters (médias por curso dentro de cada cluster) ===
                N_CURSOS   PCT  QE_I08_NUM  QE_I04_NUM  QE_I05_NUM  QE_I10_NUM  IDX_CAPITAL_CULT  NT_GER_MEDIA  TP_SEXO_PCT_9  TP_SEXO_PCT_F  TP_SEXO_PCT_M  QE_I02_PCT_A  QE_I02_PCT_B  QE_I02_PCT_C  QE_I02_PCT_D  QE_I02_PCT_E  QE_I02_PCT_F  QE_I02_PCT_NAN  NT_FG_MEDIA  NT_OBJ_FG_MEDIA  NT_DIS_FG_MEDIA  NT_CE_MEDIA  NT_OBJ_CE_MEDIA  NT_DIS_CE_MEDIA
CLUSTER_KMEANS                                                                                                                                                                                                                                                                                                                                               
0                   6242  63.6       2.779       2.497       2.884       1.487             0.480        49.721            0.0          0.768          0.232         0.491         0.079         0.017         0.317         0.003    

,NU_ANO,CO_CURSO,N_arq1,CO_IES,CO_CATEGAD,CO_ORGACAD,CO_GRUPO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO,...,QE_I22_NUM,N_arq29,QE_I23_NUM,NOME_CURSO,FAIXA_RENDA_SIMPL,IDX_CAPITAL_CULT,CATEGORIA_IES,MODALIDADE,PCT_FEMININO,CLUSTER_KMEANS
0,2023,3,33,1,1,10028,5710,1,5103403,51,...,1.451613,33,2.516129,Engenharia Civil,Média (3–6 SM),0.606048,Pública Federal,Presencial,21.2,1
1,2023,9,42,1,1,10028,17,1,5103403,51,...,1.166667,42,2.095238,Agronomia,Média (3–6 SM),0.512202,Pública Federal,Presencial,31.0,1
2,2023,10,13,1,1,10028,6405,1,5103403,51,...,1.416667,13,1.666667,Engenharia Florestal,Não informado,0.509375,Pública Federal,Presencial,69.2,0
3,2023,12,78,1,1,10028,12,1,5103403,51,...,1.358974,78,2.910256,Medicina,Não informado,0.632853,Pública Federal,Presencial,52.6,0
4,2023,16,28,1,1,10028,5806,1,5103403,51,...,1.629630,28,2.481481,Engenharia Elétrica,Média (3–6 SM),0.579167,Pública Federal,Presencial,21.4,1


### Etapa 11 - Exportação para dashboard (1 linha = 1 curso)

**Como as notas (`NT_GER`, `NT_FG`, ...) são resumidas por curso:** para cada curso/ano, calculamos a partir dos alunos com presença regular (`TP_PRES == 555`) tanto a **média** quanto a **moda** (nota mais frequente) — por isso cada `NT_*` original gera duas colunas no dashboard: `NT_GER_MEDIA` / `NT_GER_MODA`, `NT_FG_MEDIA` / `NT_FG_MODA`, etc. Ambas são arredondadas para 1 casa decimal antes de exportar.

**Como sexo, raça, renda, escolaridade dos pais, financiamento, bolsas e cotas aparecem no dashboard:** além da média/proporção usada internamente na clusterização, cada categoria de cada uma dessas variáveis gera uma coluna de **contagem absoluta** prefixada com `QTD_` — por exemplo `QTD_RACA_BRANCA` traz o número de alunos daquele curso/ano que se declararam brancos, `QTD_SEXO_FEMININO` o número de mulheres, `QTD_RENDA_ATE_1_5_SM` o número de alunos com renda familiar até 1,5 salário mínimo, e assim por diante.

**Sobre o formato do CSV:** os arquivos são salvos com `;` como separador de colunas e `,` como separador decimal — é o formato que o Excel em português reconhece corretamente. (Antes, a versão consolidada multi-ano exportava os números de nota sem arredondar, o que podia gerar valores como `4.452.727.272.727.270` ao abrir no Excel — o Excel, ao ver muitas casas decimais separadas por ponto, confundia-as com separadores de milhar. Isso foi corrigido.)

In [ ]:

COLUNAS_BASE_DASHBOARD = (
    ["NU_ANO", "CO_CURSO", "CO_IES", "CO_CATEGAD", "CATEGORIA_IES", "CO_ORGACAD",
     "CO_GRUPO", "NOME_CURSO", "CO_MODALIDADE", "MODALIDADE", "CO_UF_CURSO", "CO_REGIAO_CURSO",
     "N_ALUNOS", "PCT_FEMININO",
     "QE_I08_NUM", "FAIXA_RENDA_SIMPL", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM", "IDX_CAPITAL_CULT"]
    + VARS_DESEMPENHO_MEDIA
    + VARS_DESEMPENHO_MODA
    + ["CLUSTER_KMEANS", "CLUSTER_DBSCAN"]
)


def colunas_dashboard(df):
    """Monta a lista de colunas do dashboard: campos-base (identidade do
    curso, médias/modas de nota, índices) + TODAS as colunas de contagem
    absoluta por categoria (QTD_...) presentes no DataFrame — é assim que
    sexo, raça, renda, escolaridade dos pais, financiamento, bolsas, cotas
    etc. aparecem no resultado final."""
    base = [c for c in COLUNAS_BASE_DASHBOARD if c in df.columns]
    qtd = sorted(c for c in df.columns if c.startswith("QTD_"))
    return base + qtd


def exportar(df, ano=None):
    """Salva CSV para dashboard (1 linha = 1 curso) e Parquet com a base completa.
    Retorna (caminho_do_csv, dataframe_exportado)."""
    sfx = f"_{ano}" if ano else ""
    cols = colunas_dashboard(df)
    df_exp = df[cols].copy()

    for col in VARS_DESEMPENHO_MEDIA + VARS_DESEMPENHO_MODA:
        if col in df_exp.columns:
            df_exp[col] = df_exp[col].round(1)
    for col in ["QE_I08_NUM", "QE_I04_NUM", "QE_I05_NUM", "QE_I10_NUM", "IDX_CAPITAL_CULT", "PCT_FEMININO"]:
        if col in df_exp.columns:
            df_exp[col] = df_exp[col].round(2)

    csv_path = OUTPUTS_DIR / f"dashboard_export{sfx}.csv"
    df_exp.to_csv(csv_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    log.info(f"[Exporter] Dashboard CSV: {csv_path} ({len(df_exp):,} cursos, {len(cols)} colunas)")

    try:
        pq_path = OUTPUTS_DIR / f"enade_curso{sfx}.parquet"
        df.to_parquet(pq_path, index=False, engine="pyarrow", compression="snappy")
        log.info(f"[Exporter] Parquet (base completa): {pq_path}")
    except Exception as e:
        log.warning(f"[Exporter] Parquet não salvo (pyarrow ausente?): {e}")

    return csv_path, df_exp


print("Função de exportação definida.")


✅ Função de exportação definida.


In [15]:
# Exporta o ano de teste (usado junto com os blocos individuais acima).
# O pipeline completo multi-ano está no bloco seguinte.
csv_path, df_dashboard_teste = exportar(df_cl, ano=ANO_TESTE)

print(f"CSV exportado: {csv_path}")
print(f"Tamanho: {csv_path.stat().st_size / 1024:.0f} KB")
print(f"Colunas no dashboard: {len(df_dashboard_teste.columns)}")
print(f"Arquivos em {OUTPUTS_DIR}:")
for f in sorted(OUTPUTS_DIR.iterdir()):
    print(f"   {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

df_dashboard_teste.head(5)


[Exporter] Dashboard CSV: C:\Users\GIOVANNA\TCC_ENADE\outputs\dashboard_export_2023.csv (9,812 cursos, 109 colunas)
[Exporter] Parquet (base completa): C:\Users\GIOVANNA\TCC_ENADE\outputs\enade_curso_2023.parquet


CSV exportado: C:\Users\GIOVANNA\TCC_ENADE\outputs\dashboard_export_2023.csv
Tamanho: 3468 KB
Colunas no dashboard: 109
Arquivos em C:\Users\GIOVANNA\TCC_ENADE\outputs:
   barplot_correlacao_areas.png  (153 KB)
   correlacao_geral.csv  (1 KB)
   correlacao_por_area.csv  (1 KB)
   dashboard_export_2023.csv  (3468 KB)
   dist_notas_por_curso.png  (70 KB)
   elbow.png  (78 KB)
   enade_curso_2023.parquet  (3112 KB)
   heatmap_correlacoes_curso.png  (119 KB)
   nota_por_renda_curso.png  (49 KB)
   pca_clusters.png  (519 KB)
   perfil_clusters.csv  (1 KB)
   perfil_demografico_curso.png  (58 KB)
   perfil_por_area.csv  (1 KB)
   top_cursos_nota.png  (110 KB)


,NU_ANO,CO_CURSO,CO_IES,CO_CATEGAD,CATEGORIA_IES,CO_ORGACAD,CO_GRUPO,NOME_CURSO,CO_MODALIDADE,MODALIDADE,...,QTD_RENDA_ATE_1_5_SM,QTD_RENDA_MAIS_DE_30_SM,QTD_SEXO_FEMININO,QTD_SEXO_MASCULINO,QTD_TP_SEXO_9,QTD_TRABALHO_21_A_39H,QTD_TRABALHO_40H_OU_MAIS,QTD_TRABALHO_ATE_20H,QTD_TRABALHO_EVENTUAL,QTD_TRABALHO_NAO_TRABALHA
0,2023,3,1,1,Pública Federal,10028,5710,Engenharia Civil,1,Presencial,...,1,1,7,26,0,5,1,7,1,17
1,2023,9,1,1,Pública Federal,10028,17,Agronomia,1,Presencial,...,5,1,13,29,0,3,7,3,7,22
2,2023,10,1,1,Pública Federal,10028,6405,Engenharia Florestal,1,Presencial,...,5,0,9,4,0,3,0,0,1,8
3,2023,12,1,1,Pública Federal,10028,12,Medicina,1,Presencial,...,6,7,41,37,0,2,1,1,8,66
4,2023,16,1,1,Pública Federal,10028,5806,Engenharia Elétrica,1,Presencial,...,1,0,6,22,0,3,4,3,2,15


### Etapa 12 - Pipeline multi-ano completo
Processa **todos** os anos encontrados em `data/` (Etapa 4) e gera um CSV por ano, além de um CSV consolidado com todos os anos.

In [ ]:
import gc, traceback

resultados = []
frames_consolidado = []

for ano in ANOS_DISPONIVEIS:
    print(f"{'=' * 60}")
    print(f"  Processando {ano}...")
    print(f"{'=' * 60}")
    try:
        df_bruto_ano = ingerir_e_agregar(ano)
        df_eng_ano = adicionar_derivadas_curso(df_bruto_ano)
        del df_bruto_ano; gc.collect()

        df_cl_ano, _, _ = clusterizar(df_eng_ano, k=None, metodo="kmeans")
        del df_eng_ano; gc.collect()

        csv_ano, df_exp_ano = exportar(df_cl_ano, ano=ano)
        print(f"   {csv_ano.name}  ({csv_ano.stat().st_size / 1024:.0f} KB)  {len(df_exp_ano):,} cursos")

        # Usamos o DataFrame já arredondado/formatado (df_exp_ano) no
        # consolidado — assim a coluna de nota não acumula dezenas de casas
        # decimais (era essa a causa do número "gigante" no CSV anterior).
        frames_consolidado.append(df_exp_ano.copy())
        resultados.append({"ano": ano, "n_cursos": len(df_cl_ano)})
        del df_cl_ano; gc.collect()

    except Exception as e:
        print(f"  Falha no ano {ano}: {e}")
        traceback.print_exc()
        continue

if frames_consolidado:
    df_consolidado = pd.concat(frames_consolidado, ignore_index=True)
    consolidado_path = OUTPUTS_DIR / "dashboard_consolidado_multianos.csv"
    df_consolidado.to_csv(consolidado_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
    print(f"\nConsolidado multi-ano: {len(df_consolidado):,} linhas (curso × ano) → {consolidado_path.name}")
else:
    print("\nNenhum ano processado com sucesso.")

pd.DataFrame(resultados)


[Ingestor] ENADE 2010 — 27 arquivos disponíveis


  Processando 2010...


[Ingestor] Arquivos não encontrados (ignorados): arq28, arq29
[Ingestor] 15 arquivos agregados → 4,221 cursos × 239 colunas
[Cluster] Matriz: 4,221 cursos × 15 features
[Cluster] k ótimo: 2  Silhouettes: {2: np.float64(0.181), 3: np.float64(0.178), 4: np.float64(0.177), 5: np.float64(0.16), 6: np.float64(0.165), 7: np.float64(0.164), 8: np.float64(0.168)}
[Cluster] K-Means k=2...
[Cluster] Distribuição (cursos por cluster): {0: 2809, 1: 1412}
[Cluster] PCA: PC1=24.9%, PC2=21.9%
[Exporter] Dashboard CSV: C:\Users\GIOVANNA\TCC_ENADE\outputs\dashboard_export_2010.csv (4,221 cursos, 132 colunas)
[Exporter] Parquet (base completa): C:\Users\GIOVANNA\TCC_ENADE\outputs\enade_curso_2010.parquet
[Ingestor] ENADE 2011 — 27 arquivos disponíveis


  💾 dashboard_export_2010.csv  (1709 KB)  4,221 cursos
  Processando 2011...


[Ingestor] Arquivos não encontrados (ignorados): arq28, arq29
[Ingestor] 15 arquivos agregados → 8,814 cursos × 239 colunas
[Cluster] Matriz: 8,814 cursos × 15 features
[Cluster] k ótimo: 3  Silhouettes: {2: np.float64(0.237), 3: np.float64(0.267), 4: np.float64(0.223), 5: np.float64(0.19), 6: np.float64(0.194), 7: np.float64(0.195), 8: np.float64(0.201)}
[Cluster] K-Means k=3...
[Cluster] Distribuição (cursos por cluster): {0: 466, 1: 4383, 2: 3965}
[Cluster] PCA: PC1=34.4%, PC2=20.3%
[Exporter] Dashboard CSV: C:\Users\GIOVANNA\TCC_ENADE\outputs\dashboard_export_2011.csv (8,814 cursos, 132 colunas)
[Exporter] Parquet (base completa): C:\Users\GIOVANNA\TCC_ENADE\outputs\enade_curso_2011.parquet
[Ingestor] ENADE 2012 — 27 arquivos disponíveis


...

[Ingestor] 17 arquivos agregados → 9,812 cursos × 198 colunas
[Cluster] Matriz: 9,812 cursos × 16 features
[Cluster] k ótimo: 2  Silhouettes: {2: np.float64(0.324), 3: np.float64(0.238), 4: np.float64(0.256), 5: np.float64(0.215), 6: np.float64(0.201), 7: np.float64(0.219), 8: np.float64(0.21)}
[Cluster] K-Means k=2...
[Cluster] Distribuição (cursos por cluster): {0: 6242, 1: 3570}
[Cluster] PCA: PC1=42.7%, PC2=21.1%


  ⚠️  Falha no ano 2023: [Errno 13] Permission denied: 'C:\\Users\\GIOVANNA\\TCC_ENADE\\outputs\\dashboard_export_2023.csv'


Traceback (most recent call last):
  File "C:\Users\GIOVANNA\AppData\Local\Temp\ipykernel_30124\2824470669.py", line 26, in <module>
    csv_ano, df_exp_ano = exportar(df_cl_ano, ano=ano)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GIOVANNA\AppData\Local\Temp\ipykernel_30124\2907107574.py", line 47, in exportar
    df_exp.to_csv(csv_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
  File "C:\Users\GIOVANNA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GIOVANNA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\generic.py", line 3989, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\GIOVANNA\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_forma


✅ Consolidado multi-ano: 91,962 linhas (curso × ano) → dashboard_consolidado_multianos.csv


,ano,n_cursos
0,2010,4221
1,2011,8814
2,2012,7228
3,2013,3678
4,2014,9963
5,2015,8121
6,2016,4300
7,2017,10571
8,2018,8813
9,2019,8360
